# Keel Health - Exploratory Data Analysis (EDA)
### Dataset: UCI Diabetes 130-US Hospitals (1999–2008)
**Goal**: Understand dataset characteristics, data distributions, missing values, patient visit dynamics, and clinical leakage risks before designing preprocessing and feature engineering pipelines for the Keel Health readmission risk scoring prototype.

> **Rules**:
> - Read-only analysis. No raw data is modified or dropped.
> - No ML models trained or final feature sets created in this notebook.


## 1. Data Ingestion
Load the raw dataset using the project data-ingestion utility (`src.data.load_data.load_raw_data`).


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is accessible
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_data import load_raw_data

# Load raw dataset
df = load_raw_data()
print(f"Loaded raw dataset successfully! Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")


## 2. Dataset Dimensions & Column Profiles
Analyze the 50 columns, data types, missing counts (standard `NaN` and `?` placeholders), and unique counts.


In [2]:
# Comprehensive Column Profiler
profile_rows = []
for idx, col in enumerate(df.columns, 1):
    dtype_str = str(df[col].dtype)
    n_unique = df[col].nunique()
    nan_count = int(df[col].isna().sum())
    nan_pct = (nan_count / len(df)) * 100
    q_count = int((df[col] == '?').sum()) if dtype_str in ['object', 'str', 'string'] else 0
    q_pct = (q_count / len(df)) * 100
    total_missing = nan_count + q_count
    total_missing_pct = (total_missing / len(df)) * 100
    sample_val = str(df[col].dropna().iloc[0]) if len(df[col].dropna()) > 0 else 'None'
    
    profile_rows.append({
        "Index": idx,
        "Column": col,
        "Dtype": dtype_str,
        "Unique_Values": n_unique,
        "Standard_NaN": nan_count,
        "Question_Marks (?)": q_count,
        "Total_Missing_Pct (%)": round(total_missing_pct, 2),
        "Sample_Value": sample_val[:20]
    })

profile_df = pd.DataFrame(profile_rows)
pd.set_option('display.max_rows', 55)
profile_df


## 3. Target Distribution (`readmitted`)
Analyze the distribution of readmission classes in the raw dataset.


In [3]:
# Target Distribution Breakdown
target_counts = df['readmitted'].value_counts()
target_pct = (target_counts / len(df)) * 100
target_summary = pd.DataFrame({
    'Count': target_counts,
    'Percentage (%)': target_pct.round(2),
    'Clinical Definition': [
        'No readmission recorded within study period',
        'Readmission occurred after 30 days',
        'Early readmission (<30 days) - Primary high-risk cohort'
    ]
})
print("Target Class Distribution:")
print(target_summary)


## 4. Analysis of Missing Values & '?' Placeholders
Seven columns in the raw dataset use `'?'` to indicate missing observations. Let's quantify their missingness.


In [4]:
# Columns with '?' missingness
q_columns = {}
for col in df.columns:
    if (df[col] == '?').any():
        cnt = (df[col] == '?').sum()
        q_columns[col] = {
            'Question_Mark_Count': cnt,
            'Missing_Percentage (%)': round((cnt / len(df)) * 100, 2)
        }

q_df = pd.DataFrame(q_columns).T.sort_values(by='Missing_Percentage (%)', ascending=False)
print("Columns Containing '?' Placeholder:")
print(q_df)


## 5. Columns with Extremely High Missingness & High Cardinality
Identify columns that pose operational challenges for modeling.


In [5]:
# Extremely High Missingness (>35%)
high_missing = q_df[q_df['Missing_Percentage (%)'] > 35]
print("Columns with Extremely High Missingness (>35%):")
for col, row in high_missing.iterrows():
    print(f" - {col}: {row['Missing_Percentage (%)']}% missing")

# High Cardinality Categorical Columns
print("\nHigh Cardinality Columns (>15 unique categories):")
for col in df.select_dtypes(include=['object', 'str', 'string']).columns:
    n_uniq = df[col].nunique()
    if n_uniq > 15:
        print(f" - {col}: {n_uniq} unique categories")


## 6. Numerical Columns & Clinical Utilization Metrics
Inspect continuous/count variables indicating patient utilization and encounter intensity.


In [6]:
# Clinical Utilization & Numerical Statistics
num_cols = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures', 
    'num_medications', 'number_outpatient', 'number_emergency', 
    'number_inpatient', 'number_diagnoses'
]
df[num_cols].describe().round(2).T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]


## 7. Categorical & Medication Columns
Analyze demographic features, lab test results, and diabetic medication management.


In [7]:
# Key Demographic Categories
for demo_col in ['race', 'gender', 'age']:
    print(f"--- Distribution: {demo_col} ---")
    print(df[demo_col].value_counts().head(6))
    print()

# Lab Results (max_glu_serum, A1Cresult)
# Note: 'None' means the diagnostic test was not performed
print("--- Lab Results: max_glu_serum ---")
print(df['max_glu_serum'].value_counts(dropna=False))
print("\n--- Lab Results: A1Cresult ---")
print(df['A1Cresult'].value_counts(dropna=False))

# Diabetic Medication Changes & General Indicators
print("\n--- Medication Changes ('change') ---")
print(df['change'].value_counts())
print("\n--- Any Diabetes Medication Prescribed ('diabetesMed') ---")
print(df['diabetesMed'].value_counts())


## 8. Zero-Variance & Non-Informative Columns
Identify columns with identical values across all 101,766 encounters.


In [8]:
zero_var_cols = [col for col in df.columns if df[col].nunique() == 1]
print(f"Zero-variance columns found: {zero_var_cols}")
for col in zero_var_cols:
    print(f" - {col}: All values are '{df[col].iloc[0]}'")


## 9. Identifiers & Administrative Columns
Evaluate `encounter_id`, `patient_nbr`, and `payer_code`.


In [9]:
print(f"Total rows in dataset: {len(df):,}")
print(f"Unique encounter_ids: {df['encounter_id'].nunique():,}")
print(f"Unique patient_nbrs: {df['patient_nbr'].nunique():,}")
print(f"Payer codes unique count: {df['payer_code'].nunique()} (with {(df['payer_code'] == '?').sum():,} missing)")

# Multi-encounter patient dynamics
patient_encounter_counts = df['patient_nbr'].value_counts()
single_visit_patients = (patient_encounter_counts == 1).sum()
multi_visit_patients = (patient_encounter_counts > 1).sum()

print(f"\nPatients with exactly 1 encounter: {single_visit_patients:,} ({(single_visit_patients / len(patient_encounter_counts)) * 100:.1f}%)")
print(f"Patients with 2+ encounters:        {multi_visit_patients:,} ({(multi_visit_patients / len(patient_encounter_counts)) * 100:.1f}%)")
print(f"Maximum encounters for one patient: {patient_encounter_counts.max()}")
print(f"Total encounters from multi-visit patients: {df[df['patient_nbr'].isin(patient_encounter_counts[patient_encounter_counts > 1].index)].shape[0]:,}")


## 10. Duplicate Rows Check
Check for exact duplicates across all 50 features.


In [10]:
exact_duplicates = df.duplicated().sum()
print(f"Exact duplicate rows across all 50 columns: {exact_duplicates}")


## 11. Diagnosis Columns Investigation (`diag_1`, `diag_2`, `diag_3`)
Examine the ICD-9 code patterns across primary, secondary, and tertiary diagnoses.


In [11]:
# Top Diagnosis Codes
diag_cols = ['diag_1', 'diag_2', 'diag_3']
for col in diag_cols:
    print(f"--- Top 8 Diagnosis Codes for {col} (out of {df[col].nunique()} unique) ---")
    top_codes = df[col].value_counts().head(8)
    for code, cnt in top_codes.items():
        print(f"  Code {code:>6}: {cnt:6,} occurrences ({(cnt/len(df))*100:5.2f}%)")
    print()

# Check presence of V and E codes (supplementary classifications)
for col in diag_cols:
    v_codes = df[col].str.startswith('V').sum()
    e_codes = df[col].str.startswith('E').sum()
    q_codes = (df[col] == '?').sum()
    print(f"{col}: V-codes = {v_codes:,}, E-codes = {e_codes:,}, Missing ('?') = {q_codes:,}")


## 12. Potential Data Leakage Concerns
Investigate clinical discharge dispositions and repeat patient encounter splitting.


In [12]:
# 1. Terminal / Expired Discharge Dispositions
# discharge_disposition_id codes:
# 11 = Expired
# 13, 14 = Hospice (home / medical facility)
# 19, 20, 21 = Expired at home / medical facility / unknown
terminal_ids = [11, 13, 14, 19, 20, 21]
terminal_df = df[df['discharge_disposition_id'].isin(terminal_ids)]

print(f"Total encounters with terminal/expired discharge: {len(terminal_df):,} ({(len(terminal_df)/len(df))*100:.2f}%)")
print("Target distribution for terminal discharge patients:")
print(terminal_df['readmitted'].value_counts())
print("\nClinical Insight: Patients who expired or entered hospice cannot clinically be readmitted in standard care.")
print("Including expired patients in readmission modeling introduces severe outcome leakage and distortive bias.")


## 13. Summary & Preprocessing Recommendations for Keel Health

### A. Columns Likely Useful for Keel Health
1. **Utilization / Past History Metrics**:
   - `time_in_hospital`: Length of stay is a primary driver of post-discharge vulnerability.
   - `number_inpatient`, `number_emergency`, `number_outpatient`: Historical hospital encounters strongly correlate with recurrent readmission risk.
   - `number_diagnoses`, `num_lab_procedures`, `num_procedures`, `num_medications`: Measure overall clinical complexity and multi-morbidity burden.
2. **Clinical & Medication Factors**:
   - `admission_type_id`, `admission_source_id`: Clinical entry pathway (Emergency vs. Urgent vs. Elective).
   - `change`: Whether diabetic medications were adjusted during hospitalization (active instability).
   - `diabetesMed`: Prescribed any diabetic medication.
   - Key active medications (e.g. `insulin`, `metformin`, `glipizide`, `glyburide`).
   - `max_glu_serum` & `A1Cresult`: Glycemic control testing indicators.
3. **Primary & Secondary Diagnoses**:
   - `diag_1`, `diag_2`, `diag_3` grouped into high-level clinical disease clusters (e.g., Circulatory, Diabetes, Respiratory, Digestive, Renal).
4. **Demographics**:
   - `age` (binned age intervals), `gender`, and `race` (imputed).

---

### B. Columns Requiring Preprocessing
1. **ICD-9 Diagnosis Codes (`diag_1`, `diag_2`, `diag_3`)**: High cardinality (717–790 unique codes). Must be mapped to standardized ICD-9 disease categories (Circulatory 390–459/785, Diabetes 250, Respiratory 460–519/786, etc.).
2. **Missing Placeholders (`'?'`)**:
   - `race`: Impute with modal category or explicit `'Unknown'`.
   - `diag_1`, `diag_2`, `diag_3`: Impute minimal missing codes with `'Unknown'`.
   - `medical_specialty`: Either group into top clinical specialties (Cardiology, Internal Medicine, Surgery, Unknown) or evaluate dropping due to 49.08% missingness.
3. **Lab Result Imputation (`max_glu_serum`, `A1Cresult`)**:
   - Treat missing values (`NaN`/`None`) as a clinically informative category: `'Not Measured'`.
4. **Discharge Disposition & Admission IDs**:
   - Re-encode categorical IDs from `IDS_mapping.csv` to prevent models from treating them as continuous integers.

---

### C. Columns That May Need to Be Excluded
1. **Unique Identifiers**:
   - `encounter_id`: Primary key with 101,766 unique values (causes memorization/overfitting).
2. **Zero-Variance Features**:
   - `examide` and `citoglipton`: Contain 100% `'No'` across all 101,766 encounters.
3. **Near-Zero Variance Medications**:
   - `glimepiride-pioglitazone`, `metformin-rosiglitazone`, `metformin-pioglitazone`, `troglitazone`, `acetohexamide`: Almost 100% `'No'` (only 1–5 non-standard rows).
4. **Extreme Missingness / Administrative**:
   - `weight`: 96.86% missing (`?`).
   - `payer_code`: 39.56% missing (`?`) and represents billing/insurance administration rather than clinical pathology.

---

### D. Potential Data Leakage Concerns
1. **Terminal / Expired Discharges (`discharge_disposition_id` in [11, 13, 14, 19, 20, 21])**:
   - **Risk**: 2,423 patients expired in the hospital or were placed in terminal hospice care. They cannot be readmitted. Leaving them in the cohort creates label contamination.
   - **Remedy**: Filter out encounters where patients expired or were discharged to hospice prior to modeling.
2. **Multi-Encounter Patient Leakage (`patient_nbr`)**:
   - **Risk**: 16,773 patients have 2 or more encounters (accounting for 47,021 total encounters; up to 40 per patient). A naive random `train_test_split` will place encounters from the same patient in both train and test sets, causing severe data leakage.
   - **Remedy**: Either retain only the patient's first encounter, or enforce patient-grouped cross-validation (`GroupKFold` / `GroupShuffleSplit` on `patient_nbr`).

---

### E. Recommended Next Preprocessing Steps
1. **Cohort Selection**: Exclude terminal encounters (expired / hospice).
2. **Patient Encounter Handling**: Decide on cohort scope (first-encounter per patient vs. patient-grouped stratified split).
3. **Target Formulation**: Formulate binary classification target for the HackFest prototype (e.g., Early Readmission `<30` days vs. `NO`/`>30` days).
4. **Feature Encoding & Categorization**: Map diagnosis codes to disease families, one-hot encode categorical features, scale numerical features.
